# 11 · Hugging Face corpora — Lucius-Morningstar

Navigate and explore the datasets the mailroom family publishes on
[Lucius-Morningstar](https://huggingface.co/Lucius-Morningstar). The
targeted full corpus is **`mailroom-dataset` v9** (3,302 docs).
Class × subtype **examples** come from **`docclass-pilot`** (48 strata).
Other pipeline-ready Hub sets (Enron correspondence ~247k, CMS claims,
CUAD contracts) ingest the same way; `legalbench-full` is a LegalBench
CLI task pack, not a document-pipeline ingest.

**What you'll see:** the org catalog, the committed class×subclass example
pack, first-row previews, substring search, an equality filter, and one
Hub row fed into the real pipeline (mock LLM).

**Honesty label:** default cells are OFFLINE. They read a committed Dataset
Viewer snapshot under `notebooks/fixtures/huggingface/` (dated in
`catalog.json`). Nothing below talks to the Hub unless you set
`MAILROOM_HF_LIVE=1` and run the marker-gated live cell at the bottom.
`legalbench-full` on the Hub is currently a stub viewer (placeholder rows);
notebook 12 is the real LegalBench suite.

Companion: `notebooks/huggingface_lab.py`.


## Setup


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "notebooks" / "pipeline_lab.py").exists(), (
    f"llm-mailroom repo root not found above {Path.cwd()}"
)
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

import huggingface_lab as hf
import pipeline_lab as lab
lab.quiet_logs()
print("snapshot date:", hf.catalog()["snapshot_date"])
print("source:", hf.catalog()["source"] if "source" in hf.catalog() else "offline-snapshot")
print("live requested:", hf.live_requested())


## The catalog

Seven datasets. `mailroom_classes` is the wiring back to `taxonomy.yaml`,
not a Hub tag — it is how this pipeline consumes the published surface.
`mailroom-dataset` v9 is the full corpus; `docclass-pilot` is one example
of every type and subtype. `compliance_filing` has zero Hub rows.


In [ ]:
cat = hf.catalog()
hf.show_catalog(cat["datasets"])
print()
print("org:", cat["org_url"])


## Class × subclass examples (docclass-pilot, v5 parent)

One Hub row per stratum — every type and subtype in `mailroom-dataset` v9.
This pack is what `--mock` / `--examples` on `run_hf_pilot.py` and the
notebook `CLASS_PACKS` use. Not invented stand-in text.


In [ ]:
pack = hf.class_subclass_examples()
print("parent:", pack["parent"], "schema:", pack["schema"], "strata:", pack["n_strata"])
from collections import Counter
print(Counter(row["expected"] for row in pack["examples"]))
print("sample:", pack["examples"][0]["filename"], "/", pack["examples"][0]["expected_subclass"])


## Preview — insurance claims (CMS DE-SynPUF)

Synthetic Medicare claims, labeled `insurance_claim`. No real PHI.


In [ ]:
claims = hf.preview("Lucius-Morningstar/cms-desynpuf-insurance-claims", length=3)
hf.show_rows(claims, text_chars=140)
print("features:", claims.get("features"))


## Preview — Enron correspondence (deduped)


In [ ]:
enron = hf.preview("Lucius-Morningstar/enron-correspondence-dedup", length=3)
hf.show_rows(enron, text_chars=120)


## Preview — CUAD full contracts (clause labels in `expected`)


In [ ]:
cuad = hf.preview("Lucius-Morningstar/mailroom-cuad-contracts-full", length=2)
hf.show_rows(cuad, text_chars=140)


## Search the snapshot

Offline search is a substring over the committed first-row window — not the
full 247k Enron rows. The live cell at the bottom uses Dataset Viewer `/search`.


In [ ]:
hits = hf.search("Lucius-Morningstar/enron-correspondence-dedup", "forecast")
print("query=forecast  hits=", len(hits["rows"]), " source=", hits["source"])
hf.show_rows(hits, text_chars=100)


## Filter — insurance rows labeled `insurance_claim`


In [ ]:
filt = hf.filter_rows(
    "Lucius-Morningstar/cms-desynpuf-insurance-claims",
    where="expected=insurance_claim",
)
print("hits:", len(filt["rows"]), "source:", filt["source"])
print("labels:", sorted({r.get("expected") for r in filt["rows"]}))


## Feed a Hub row into the pipeline

Take the first DE-SynPUF claim's `doc_text`, run it as an `insurance_claim`
through the real graph (mock specialist). The catalog join in
`dataset_browser` is the local-pilot analogue of this.


In [ ]:
row = claims["rows"][0]
text = hf.row_to_doc_text(row)
print("filename:", row.get("filename"), " chars:", len(text))
env = lab.open_sandbox()
lab.script_all_specialists(env["client"])
run = lab.run_document(
    env, text[:4000], filename="hf_claim.txt",
    classification=lab.CLASSIFY_INSURANCE_HIGH,
    extraction=lab.INSURANCE_CLAIM_EXTRACTION,
)
print("path:", " → ".join(lab.path_of(run["steps"])))
print("stage:", run["final"].get("stage"), " type:", run["final"].get("doc_type"))
lab.close_sandbox(env)


## Docclass + CUAD + LegalBench at a glance

| dataset | mailroom use |
|---|---|
| `mailroom-dataset` (v9, 3,302 docs) | targeted full pipeline corpus |
| `docclass-pilot` (48 strata) | class × subclass examples |
| `mailroom-cuad-contracts` | vision surface (page images) |
| `mailroom-cuad-contracts-full` | contract texts + CUAD clause labels |
| `legalbench-full` | LegalBench CLI tasks — not pipeline ingest |
| `enron-correspondence-dedup` (~247k) | correspondence specialist |
| `cms-desynpuf-insurance-claims` | insurance_claim specialist |

Committed PDFs under `docs/examples/samples/` are PDF-ingest fixtures, not the class catalog. `dataset_browser.ipynb` still walks that local set.


## Live Hub refresh (opt-in)

<!-- NB-OPT-IN-NETWORK: Dataset Viewer / Hub API; skipped unless MAILROOM_HF_LIVE=1 -->

Set `MAILROOM_HF_LIVE=1` (optional `HF_TOKEN` for higher rate limits) and
re-run to hit `https://datasets-server.huggingface.co` for a fresh catalog
and a live search. Default execution never takes this branch.


In [ ]:
import os
if os.environ.get("MAILROOM_HF_LIVE", "").strip().lower() in ("1", "true", "yes", "on"):
    live_cat = hf.catalog(live=True)
    print("LIVE catalog source:", live_cat.get("source"), "n=", len(live_cat["datasets"]))
    live_hits = hf.search("Lucius-Morningstar/enron-correspondence-dedup", "forecast", live=True)
    print("LIVE search hits:", live_hits.get("num_rows_total"), "source:", live_hits.get("source"))
else:
    print("live cell skipped (MAILROOM_HF_LIVE not set) — offline snapshot used above.")


## Where to go next

- **12 legalbench** — the in-repo eval suite (binary QA + family classification)
- **13 vision_ingestion** — page images, the other CUAD surface
- **dataset_browser** — the 30 local pilot samples + catalog overlay
